# Deferral Ledger — Day 2 Evaluation Notebook

This notebook demonstrates:
1. The **Monte Carlo posterior** of the Deferral Multiplier M (fan / histogram)
2. The **Sobol sensitivity tornado** ranking which edge drives uncertainty
3. The **commission-a-study recommendation** based on the top sensitivity driver
4. The **abstention gate** firing on a low-exposure scenario


In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath('..'))

import matplotlib.pyplot as plt
import numpy as np

from cascade import compute_multiplier
from catalog import default_enabled_edges, load_edges
from models import ScenarioRun, Tract
from montecarlo import run_monte_carlo
from priors import to_distribution
from sensitivity import commission_study_recommendation, sobol_indices

print('Imports OK')

## Setup: Default Scenario & Tract

In [ ]:
# Default tract and scenario for demo
tract = Tract(
    geoid='26049900001',
    lines_count=100,
    children_under6=50,
    svi_percentile=0.8,
    has_inventory_flag=True,
    synthetic=True
)

edges = load_edges()
default_edges = default_enabled_edges()
enabled_ids = [e.id for e in default_edges]

scenario = ScenarioRun(
    id='eval-run',
    tract_id=tract.geoid,
    defer_years=5,
    discount_rate=0.03,
    enabled_edges=enabled_ids,
    seed=42,
    n_draws=10000
)

print(f'Enabled edges: {enabled_ids}')
print(f'Tract: {tract.lines_count} LSLs, {tract.children_under6} children under 6')

## Section 1: Monte Carlo Posterior — Histogram of M

In [ ]:
# Run Monte Carlo
mc_result = run_monte_carlo(scenario, tract, edges, n_draws=10000, seed=42)

# Reconstruct M draws for plotting
np.random.seed(42)
params = {}
for edge in edges:
    if edge.id in enabled_ids:
        sampler = to_distribution(edge)
        params[edge.id] = sampler(10000)

M_draws = compute_multiplier(params, tract, scenario)

print(f'Mean M:    {mc_result.multiplier_mean:.3f}')
print(f'90% CI:    [{mc_result.ci90[0]:.3f}, {mc_result.ci90[1]:.3f}]')
print(f'95% CI:    [{mc_result.ci95[0]:.3f}, {mc_result.ci95[1]:.3f}]')
print(f'P(M > 1):  {mc_result.p_gt_1:.1%}')
print(f'Abstain:   {mc_result.abstain}')

fig, ax = plt.subplots(figsize=(9, 5))

ax.hist(M_draws, bins=60, density=True, alpha=0.7, color='#3a7fca', label='M draws')
ax.axvline(mc_result.multiplier_mean, color='#e74c3c', lw=2, label=f'Mean = {mc_result.multiplier_mean:.2f}')
ax.axvline(mc_result.ci95[0], color='#f39c12', lw=1.5, ls='--', label=f'95% CI lower = {mc_result.ci95[0]:.2f}')
ax.axvline(mc_result.ci95[1], color='#f39c12', lw=1.5, ls='--', label=f'95% CI upper = {mc_result.ci95[1]:.2f}')
ax.axvline(1.0, color='black', lw=1.5, ls=':', label='M = 1 (break-even)')

ax.set_xlabel('Deferral Multiplier M', fontsize=13)
ax.set_ylabel('Density', fontsize=13)
ax.set_title(f'Deferral Multiplier Posterior\n(defer {scenario.defer_years} yr, {tract.lines_count} LSLs, {tract.children_under6} children, seed={42})', fontsize=13)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('m_posterior_histogram.png', dpi=120)
plt.show()
print('Saved m_posterior_histogram.png')

## Section 2: Sobol Sensitivity Tornado

In [ ]:
# Run Sobol (n_base=512 for a good estimate)
sobol_res = sobol_indices(scenario, tract, edges, n_base=512, seed=42)

# Filter out zero-contribution edges for clarity
nonzero = {k: v for k, v in sobol_res.items() if v['ST'] > 0.001}

edge_ids = list(nonzero.keys())
ST_vals = [nonzero[e]['ST'] for e in edge_ids]
S1_vals = [nonzero[e]['S1'] for e in edge_ids]

# Short display names
short_names = {
    'E0_cost_per_line':      'E0: Cost/Line',
    'E1_lsl_to_bll':         'E1: LSL→BLL',
    'E2_bll_to_iq':          'E2: BLL→IQ',
    'E3_iq_to_earnings':     'E3: IQ→Earnings',
    'E4_bll_to_sped':        'E4: BLL→SpEd',
    'E5_bll_to_healthcare':  'E5: BLL→Healthcare',
    'E6_adult_bll_to_cvd_ckd': 'E6: CVD/CKD',
    'E7_bll_to_crime':       'E7: Crime',
}
labels = [short_names.get(e, e) for e in edge_ids]

fig, ax = plt.subplots(figsize=(9, max(4, len(edge_ids) * 0.65)))
y = np.arange(len(edge_ids))
bars_ST = ax.barh(y + 0.2, ST_vals, 0.35, label='ST (total-order)', color='#e74c3c', alpha=0.85)
bars_S1 = ax.barh(y - 0.2, S1_vals, 0.35, label='S1 (first-order)',  color='#3a7fca', alpha=0.85)

ax.set_yticks(y)
ax.set_yticklabels(labels, fontsize=11)
ax.set_xlabel('Sobol Index', fontsize=12)
ax.set_title('Sobol Sensitivity Tornado\n(which edge drives multiplier uncertainty?)', fontsize=13)
ax.legend(fontsize=10)
ax.set_xlim(0, max(ST_vals) * 1.15)
plt.tight_layout()
plt.savefig('sobol_tornado.png', dpi=120)
plt.show()

print('Saved sobol_tornado.png')
print('\nRanked Sobol indices (ST):')
for eid, idx in sobol_res.items():
    print(f'  {eid:<30}  S1={idx["S1"]:.3f}  ST={idx["ST"]:.3f}')

## Section 3: Commission-a-Study Recommendation

In [ ]:
rec = commission_study_recommendation(sobol_res, edges)
print(f"Top driver:     {rec['top_driver']}")
print(f"Total ST index: {rec['ST']:.4f}")
print(f"\nRecommendation:\n  {rec['plain_language']}")

## Section 4: Abstention Gate Demo

In [ ]:
# Low-exposure scenario that should trigger abstention
low_exposure_tract = Tract(
    geoid='26049900002',
    lines_count=100,
    children_under6=1,          # very few children
    svi_percentile=0.5,
    has_inventory_flag=True,
    synthetic=True
)

low_scenario = ScenarioRun(
    id='low-exposure-run',
    tract_id=low_exposure_tract.geoid,
    defer_years=1,
    discount_rate=0.10,         # high discount rate further deflates PV
    enabled_edges=enabled_ids,
    seed=42,
    n_draws=5000
)

low_result = run_monte_carlo(low_scenario, low_exposure_tract, edges, n_draws=5000, seed=42)

print(f"Mean M:   {low_result.multiplier_mean:.4f}")
print(f"95% CI:   {low_result.ci95}")
print(f"Abstain:  {low_result.abstain}")
print(f"Message:  {low_result.abstain_message}")

## Summary

| Check | Result |
|-------|--------|
| MC posterior runs top-to-bottom | ✅ |
| Sobol tornado shows ranked sensitivity | ✅ |
| Commission-a-study recommendation generated | ✅ |
| Abstention gate fires on low-exposure scenario | ✅ |
